# Embedding the OCR corpus

Build a persistent, searchable index from the Unlimited-OCR Markdown and layout sidecars in `assets/pdf.remaining/.md_unlimited_v2`. Chroma stores passage and section embeddings. SQLite stores exact OCR text, lexical search data, and the source locations used for citations.

Run the cells in order using the repository's Python environment and an Ollama server with the embedding model installed. On a fresh checkout, run `nbdev-export` from the repository root to generate the project's notebook-backed modules.

In [ ]:
#| default_exp core.embedding.ocr

In [ ]:
#| export
import json
from pathlib import Path

from ribosome.preprocessing.ocr.layout_bundle import ingest_layout_bundle, pair_layout_bundles
from ribosome.retrieval.layout_rag import HybridIndexer, OllamaEmbeddingProvider
from tqdm.auto import tqdm

## Discover and index OCR bundles

Pair each published sidecar with its Markdown file and, when available, the PDF at the same relative path. The layout reader validates the pair, groups regions into passages and table rows, and retains page numbers and bounding boxes. Embeddings use OCR text with section context. The PDF supplies a citation target.

In [ ]:
#| export
def discover_ocr_bundles(ocr_root):
    root = Path(ocr_root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"OCR directory does not exist: {root}")
    def published(path):
        return all(not (part.startswith(".") or part.endswith(".assets"))
                   for part in path.relative_to(root).parts)

    bundles = [
        bundle for bundle in pair_layout_bundles(root, pdf_roots=[root.parent])
        if published(bundle.layout_path)
    ]
    markdown = {p for p in root.rglob("*.md") if published(p) and not p.name.endswith(".partial.md")}
    unpaired = markdown - {bundle.markdown_path for bundle in bundles}
    if unpaired:
        raise ValueError(f"Missing layout sidecar for {min(unpaired)}")
    if not bundles:
        raise ValueError(f"No published OCR layout bundles found under {root}")
    return bundles


def batched_embedder(client, model, batch_size=16):
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    provider = OllamaEmbeddingProvider(client, model)

    async def embed(texts):
        vectors = []
        for start in range(0, len(texts), batch_size):
            vectors.extend(await provider(texts[start:start + batch_size]))
        return vectors

    return embed

In [ ]:
#| export
def _already_indexed(store, dense, result):
    document = result.document
    row = store.connection.execute(
        "SELECT artifact_hashes_json, metadata_json FROM documents "
        "WHERE document_id = ? AND pipeline_version = ?",
        (document.document_id, document.pipeline_version),
    ).fetchone()
    if row is None or json.loads(row["artifact_hashes_json"]) != document.artifact_hashes:
        return False
    metadata = json.loads(row["metadata_json"])
    if metadata["layout_path"] != str(document.paths.layout_path):
        return False
    where = {"document_id": document.document_id}
    child_ids = {record.chunk_id for record in result.indexable_records}
    parent_ids = {p.parent_id for p in result.sections if p.summary and not p.is_toc}
    return (
        set(dense.children.get(where=where, include=[])["ids"]) == child_ids
        and set(dense.parents.get(where=where, include=[])["ids"]) == parent_ids
    )


async def index_ocr_corpus(ocr_root, store, dense, embedder):
    '''Synchronize a dedicated OCR index, resuming from completed documents.'''
    bundles = discover_ocr_bundles(ocr_root)
    indexer = HybridIndexer(store, dense)
    current_ids = set()
    indexed = reused = 0
    for bundle in tqdm(bundles, desc="OCR documents", unit="doc"):
        try:
            result = ingest_layout_bundle(
                bundle.layout_path, markdown_path=bundle.markdown_path, pdf_path=bundle.pdf_path,
                pipeline_version=dense.pipeline_version, reconcile_native_text=False,
            )
            if result.document.status not in {"completed", "processed"}:
                raise ValueError(f"Incomplete OCR bundle ({result.document.status})")
            current_ids.add(result.document.document_id)
            if _already_indexed(store, dense, result):
                reused += 1
                continue
            await indexer.index(result, embedder=embedder)
            indexed += 1
        except Exception as error:
            raise RuntimeError(f"Could not index {bundle.layout_path}: {error}") from error

    # Only retire obsolete records once every current bundle has succeeded.
    previous_ids = {row[0] for row in store.connection.execute(
        "SELECT document_id FROM documents WHERE pipeline_version = ?", (dense.pipeline_version,)
    )}
    for obsolete in previous_ids - current_ids:
        dense.delete_document(obsolete)
        store.delete_document(obsolete, pipeline_version=dense.pipeline_version)
    return {"indexed": indexed, "reused": reused, "documents": len(current_ids),
            "passage_vectors": dense.children.count(), "section_vectors": dense.parents.count()}

## Configure the database

The default endpoint is the project's embedding server. Set `OLLAMA_EMBED_HOST` or `OLLAMA_EMBED_MODEL` to override it. The database directory is dedicated to this corpus, model, and layout pipeline version. Rerunning checks source hashes and vector IDs before requesting embeddings. A successful corpus pass also removes records for sources that have left this corpus.

In [ ]:
#| notest
import os
from contextlib import asynccontextmanager, closing
from hashlib import sha256

from chromadb import PersistentClient
from dotenv import load_dotenv
from ollama import AsyncClient

from ribosome.application.layout_rag import LayoutRAGApplication
from ribosome.ollama_config import OLLAMA_EMBED_HOST, OLLAMA_TRUST_ENV
from ribosome.preprocessing.ocr.layout_bundle import LAYOUT_PIPELINE_VERSION
from ribosome.retrieval.layout_rag import ChromaDenseIndex, SQLiteRetrievalRecordStore

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file())
load_dotenv(PROJECT_ROOT / ".env", override=False)
OCR_ROOT = PROJECT_ROOT / "assets/pdf.remaining/.md_unlimited_v2"
MODEL = os.getenv("OLLAMA_EMBED_MODEL") or "qwen3-embedding:8b"
HOST = os.getenv("OLLAMA_EMBED_HOST") or OLLAMA_EMBED_HOST
BATCH_SIZE = 16

namespace = json.dumps([str(OCR_ROOT.resolve()), MODEL, LAYOUT_PIPELINE_VERSION])
INDEX_DIR = PROJECT_ROOT / "db/ocr_embeddings" / sha256(namespace.encode()).hexdigest()[:16]
print(f"Corpus: {OCR_ROOT}\nEmbedding model: {MODEL}\nDatabase: {INDEX_DIR}")

In [ ]:
#| notest
@asynccontextmanager
async def open_ocr_index():
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    key = os.getenv("OLLAMA_API_KEY")
    client = AsyncClient(host=HOST, headers={"Authorization": f"Bearer {key}"} if key else {},
                         trust_env=OLLAMA_TRUST_ENV, timeout=300)
    try:
        with closing(PersistentClient(path=str(INDEX_DIR / "chroma"))) as chroma:
            with SQLiteRetrievalRecordStore(INDEX_DIR / "records.sqlite3") as store:
                dense = ChromaDenseIndex(chroma, embedding_model=MODEL, collection_prefix="ocr")
                yield LayoutRAGApplication(store, dense_index=dense,
                                           embedder=batched_embedder(client, MODEL, BATCH_SIZE))
    finally:
        await client.close()

## Build or resume the index

In [ ]:
#| notest
async with open_ocr_index() as app:
    counts = await index_ocr_corpus(OCR_ROOT, app.store, app.dense_index, app.embedder)
print(counts)

## Retrieve cited evidence

After a build, run the configuration cells and this query to search the saved database. Each hit includes its source Markdown path, page and region citation, and an excerpt of the exact text.

In [ ]:
#| notest
question = "新松机器人急停后如何恢复运行？"
async with open_ocr_index() as app:
    matches = await app.query(question, limit=3)
if not matches.evidence:
    print("No matching OCR evidence found.")
for item in matches.evidence:
    print(item.citation, item.hit.record.source_markdown_uri, sep="\n")
    print(item.exact_text[:600], end="\n\n")